# LightGBM Training and Macro F1 Optimization

This notebook trains the final LightGBM classifier for predictive maintenance, using the validated pipeline from previous notebooks: the fused dataset (sensor + rolling features + external context), 5-fold stratified cross-validation, and SMOTE applied only inside training folds. The target metric is **Macro F1 ≥ 0.85**, as specified in the project KPIs.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

In [ ]:
df = pd.read_csv("../data/processed/fused_dataset.csv")

target = 'Machine failure'
exclude_cols = ['UDI', 'Product ID', 'Type', 'timestamp', target,
                'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].fillna(0)
y = df[target]

print("Features:", len(feature_cols))
print("Failure rate:", round(y.mean()*100, 2), "%")

In [ ]:
# LightGBM doesn't allow special JSON characters like [ ] in feature names
import re

def clean_column_name(col):
    return re.sub(r'[^\w]+', '_', col).strip('_')

X.columns = [clean_column_name(c) for c in X.columns]
print("Cleaned feature names:")
print(X.columns.tolist())

## Training pipeline: 5-fold CV + SMOTE (train fold only) + LightGBM

Each fold follows the leakage-free SMOTE process validated in Issue #8, now using LightGBM as the classifier.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []
fold_models = []
all_y_test = []
all_preds = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=31,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train_res, y_train_res)
    preds = model.predict(X_test)

    score = f1_score(y_test, preds, average='macro')
    fold_scores.append(score)
    fold_models.append(model)
    all_y_test.extend(y_test)
    all_preds.extend(preds)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\nMean Macro F1: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
print(f"Target (>= 0.85): {'PASSED' if np.mean(fold_scores) >= 0.85 else 'NOT YET MET'}")

In [ ]:
print("Classification Report (5-fold out-of-fold predictions):")
print(classification_report(all_y_test, all_preds, target_names=['No Failure', 'Failure']))

In [ ]:
cm = confusion_matrix(all_y_test, all_preds)
print("Confusion Matrix:")
print(f"                 Predicted No-Fail  Predicted Fail")
print(f"Actual No-Fail        {cm[0][0]:>6}            {cm[0][1]:>6}")
print(f"Actual Fail           {cm[1][0]:>6}            {cm[1][1]:>6}")

In [ ]:
best_fold_idx = np.argmax(fold_scores)
best_model = fold_models[best_fold_idx]

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"Top 10 most important features (from best fold, Fold {best_fold_idx+1}):")
print(importance_df.head(10).to_string(index=False))

In [ ]:
import joblib

joblib.dump(best_model, "../models/lightgbm_predictive_maintenance.pkl")
print("Best model saved to: ../models/lightgbm_predictive_maintenance.pkl")

## Summary

The LightGBM classifier, trained with 5-fold stratified cross-validation and SMOTE applied only within training folds, achieved a **Mean Macro F1 of 0.9136 (+/- 0.0038)** — comfortably exceeding the project's target of ≥ 0.85. This outperforms the earlier Random Forest + SMOTE baseline (0.8718) from Issue #8, confirming LightGBM as the stronger choice of classifier for this imbalanced predictive maintenance task. The best-performing fold's model is saved for use in subsequent noise robustness testing (Issue #10) and threshold tuning (Issue #11).